In [ ]:
import os
import requests
import pandas as pd
import xml.etree.ElementTree as ET
import time
from tqdm import tqdm
from dotenv import load_dotenv  # 추가: .env 파일 로드를 위한 라이브러리

# ==========================================
# 0. 환경 변수 로드 (API Key 관리)
# ==========================================
load_dotenv()  # .env 파일의 내용을 환경 변수로 불러옵니다.
API_KEY = os.getenv("KCI_API_KEY")

if not API_KEY:
    raise ValueError("API_KEY를 찾을 수 없습니다. .env 파일에 'KCI_API_KEY'가 설정되어 있는지 확인하세요.")

# ==========================================
# 1. 설정 및 데이터 로드
# ==========================================
INPUT_FILE = '법학 기관 목록.csv'
OUTPUT_FILE = '전체_법학_논문목록.csv'
BASE_URL = "https://open.kci.go.kr/po/openapi/openApiSearch.kci"

try:
    df_org = pd.read_csv(INPUT_FILE)
    target_institutions = df_org['기관명'].dropna().unique().tolist()
    print(f"총 {len(target_institutions)}개의 기관을 대상으로 수집을 시작합니다.")
except Exception as e:
    print(f"파일 로드 중 오류 발생: {e}")
    target_institutions = []

# ==========================================
# 2. 수집 함수 정의 (동일)
# ==========================================
def fetch_articles_by_institution(api_key, institution_name):
    articles = []
    page = 1
    display_count = 100
    
    while True:
        params = {
            "apiCode": "articleSearch",
            "key": api_key,
            "institution": institution_name,
            "displayCount": display_count,
            "page": page
        }
        
        try:
            response = requests.get(BASE_URL, params=params)
            if response.status_code != 200:
                print(f"Error: {response.status_code} for {institution_name}")
                break
                
            root = ET.fromstring(response.content)
            total_count_node = root.find(".//outputData/result/total")
            if total_count_node is None:
                break
            total_count = int(total_count_node.text)
            
            if total_count == 0:
                break
                
            records = root.findall(".//outputData/record")
            if not records:
                break
                
            for record in records:
                article_info = record.find("articleInfo")
                journal_info = record.find("journalInfo")
                
                if article_info is not None:
                    article_id = article_info.get("article-id")
                    title_group = article_info.find("title-group")
                    article_title = ""
                    if title_group is not None:
                        title_node = title_group.find("article-title[@lang='original']")
                        if title_node is not None:
                            article_title = title_node.text
                        else:
                            first_title = title_group.find("article-title")
                            article_title = first_title.text if first_title is not None else ""

                    author_group = article_info.find("author-group")
                    authors = []
                    if author_group is not None:
                        for author in author_group.findall("author"):
                            if author.text:
                                authors.append(author.text.strip())
                    
                    pub_year = ""
                    if journal_info is not None:
                        pub_year_node = journal_info.find("pub-year")
                        pub_year = pub_year_node.text if pub_year_node is not None else ""
                    
                    articles.append({
                        "institution": institution_name,
                        "article_id": article_id,
                        "title": article_title,
                        "authors": ", ".join(authors),
                        "pub_year": pub_year,
                        "kci_url": article_info.find("url").text if article_info.find("url") is not None else ""
                    })
            
            if page * display_count >= total_count:
                break
            page += 1
            time.sleep(0.5)
            
        except Exception as e:
            print(f"Error parsing {institution_name} page {page}: {e}")
            break
            
    return articles

# ==========================================
# 3. 실행 및 저장
# ==========================================
all_results = []

for inst in tqdm(target_institutions, desc="기관별 논문 수집 중"):
    results = fetch_articles_by_institution(API_KEY, inst)
    all_results.extend(results)
    time.sleep(0.5)

if all_results:
    df_results = pd.DataFrame(all_results)
    df_results.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
    print(f"\n수집 완료! '{OUTPUT_FILE}' 파일에 총 {len(df_results)}건의 논문 정보가 저장되었습니다.")
else:
    print("수집된 데이터가 없습니다.")